In [ ]:
import hashlib
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"               # no coding-agent hint in the outputs
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"  # no download progress bars
import numpy as np
import pandas as pd
from tqdm import TqdmWarning

warnings.filterwarnings("ignore", message="IProgress not found", category=TqdmWarning)  # as in notebook 04

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
GOLD_DB = PROJECT_ROOT / "data" / "gold" / "fraud_dev.duckdb"
MLFLOW_DB = PROJECT_ROOT / "mlflow.db"
MASTER_SEED = 42
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def git(*args):
    return subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True, text=True).stdout.strip()


assert ".venv" in sys.executable, f"not the project environment: {sys.executable}"
MLFLOW_SHA_BEFORE = sha256(MLFLOW_DB)
T_START = time.perf_counter()
print("python:", sys.executable)
print("git commit:", git("rev-parse", "--short", "HEAD"), "| uncommitted changes:", bool(git("status", "--porcelain")))
print("master seed:", MASTER_SEED)
print("mlflow.db SHA-256 before:", MLFLOW_SHA_BEFORE)

# Phase 5: A/B experiment design and causal inference for step-up verification

**Every treatment effect in this notebook is a SIMULATED ASSUMPTION, not a measurement.** The value of this phase is the design and the analysis pipeline, not a finding about real customers.

- **The model is frozen.** Final model F (LightGBM, 16 features), loaded exactly as logged in MLflow (read only). It is not retrained or changed, no MLflow run is created or modified, and the SHA-256 of `mlflow.db` is checked before and after.
- **Periods.** Pre-period 2020-07-01 to 2020-09-30 (92 days) and experiment period 2020-10-01 to 2020-12-31 (92 days, the test period, already opened once in Phase 4). F was trained up to 2020-06-23, so its scores are out of sample in both. The pre-period overlaps the validation period on which the Platt calibrator was fitted; this notebook uses F's raw score, and that calibrator is close to the identity (a = 0.995, b = −0.086), so the effect is negligible, but it is stated here. No decision about F is taken from Phase 5.
- **Units.** Randomisation unit and analysis unit are the card: one card's transactions are not independent (one fraud episode spans several of them).

## Pre-registered decision rule (written before any simulated result)

**Experiment.** Step-up verification for every transaction whose F score is at least **0.02**. Cards are assigned to treatment or control by an independent coin flip (probability 0.5). Fixed horizon of 92 days, no interim looks.

**Population.** Primary: **triggered cards**, i.e. cards with at least one transaction scored at or above the threshold. A card enters at its first such transaction, computed identically in both arms from the logged F scores (F is frozen; the control arm logs the score without showing a step-up). Secondary check: all enrolled cards (intention to treat).

**Metrics** (per triggered card, control minus treatment, so positive = treatment helps):
- **Primary (the single confirmatory test): fraud loss per card**, two-sided Welch test, α = 0.05.
- **Decision metric: total loss per card** = fraud loss + m × abandoned legitimate dollars, with m = 1.0 (the full transaction value counts as lost).
- **Guardrails** (one-sided harm tests, Holm-adjusted over the two, α = 0.05):
  - G1, lost legitimate revenue per card: a breach if the increase is larger than 5% of the control arm's mean legitimate dollars stepped up per card;
  - G2, abandonment rate of stepped-up legitimate transactions (pooled, delta method): a breach if it is above 10%.
- **Sample-ratio mismatch (SRM):** chi-square test on the number of cards per arm; p < 0.001 makes the experiment invalid.

**Decision.** **Ship** only if there is no SRM (p ≥ 0.001), the 95% interval of (control − treatment) total loss has a lower bound above 0, and no Holm-adjusted guardrail is breached. **Don't ship** otherwise. **Extend** only if enrolment fell short of the plan with no SRM and no breach (cannot happen in this simulation).

**Simulated scenarios (assumptions, not findings).** Share of stepped-up fraud the step-up blocks b ∈ {0.5, 0.65, 0.8}; share of stepped-up legitimate transactions abandoned a ∈ {1%, 5%, 10%, 20%}; m = 1.0, with m = 0.25 and 0.02 as sensitivity; thresholds 0.5, 0.294, 0.1 and 0.005 as sensitivity only. 500 simulated experiments per primary scenario. Master seed 42.

**Stop conditions.** Any reconciliation mismatch; a full-pipeline A/A false-positive rate outside 0.035–0.065; an SRM failure in the A/A; a changed `mlflow.db`; a runtime above 30 minutes.

In [ ]:
# Load the Gold data, rebuild F's features (past-only) and score both periods with F exactly as logged
import sqlite3

import mlflow.lightgbm

from src.models.evaluate import alert_metrics, top_n_mask
from src.models.features import build_features, load_model_frame
from src.models.train import FINAL_COLUMNS

t0 = time.perf_counter()
frame = load_model_frame(GOLD_DB)
feats = build_features(frame)
con = sqlite3.connect(f"file:{MLFLOW_DB.as_posix()}?mode=ro", uri=True)
f_run = con.execute("select run_uuid from runs where name = 'F_coarse_profile' and lifecycle_stage = 'active'").fetchall()
assert len(f_run) == 1, f_run
f_dir = con.execute("select artifact_location from logged_models where source_run_id = ?", f_run[0]).fetchone()[0]
con.close()
f_model = mlflow.lightgbm.load_model(f_dir)

ts = pd.to_datetime(frame["trans_ts"])
PRE = ((ts >= "2020-07-01") & (ts < "2020-10-01")).to_numpy()
EXP = ((ts >= "2020-10-01") & (ts <= "2020-12-31 23:59:59")).to_numpy()
score = np.full(len(frame), np.nan)
score[PRE | EXP] = f_model.predict_proba(feats.loc[PRE | EXP, list(FINAL_COLUMNS)])[:, 1]
tx = pd.DataFrame({"card_key": frame["card_key"], "ts": ts, "amt": frame["amt"],
                   "is_fraud": frame["is_fraud"].astype(int), "score": score, "gender": frame["gender"],
                   "age_band": feats["age_band"], "urban": feats["urban"]})
tx_pre, tx_exp = tx[PRE], tx[EXP]
DAYS = 92
print(f"F run {f_run[0][0][:8]} loaded from its artifact folder; scored in {time.perf_counter() - t0:.0f} s")
for name, d in [("pre-period", tx_pre), ("experiment period", tx_exp)]:
    print(f"{name}: {len(d):,} transactions, {d.is_fraud.sum()} fraud, {d.card_key.nunique()} active cards, "
          f"{d.loc[d.is_fraud == 1, 'card_key'].nunique()} fraud cards")

In [ ]:
# Reconciliation 1 (deterministic, must match the design numbers exactly; stops on any mismatch)
from src.experiments.ab_design import per_card_metrics, required_n, threshold_table, variance_reduction

problems = []
def check(label, got, expected):
    ok = got == expected
    print(f"{'ok  ' if ok else 'FAIL'} {label}: {got} (expected {expected})")
    if not ok:
        problems.append(label)

pc = alert_metrics(tx_exp["is_fraud"].to_numpy(), top_n_mask(tx_exp["score"].to_numpy(), 1082), tx_exp["amt"].to_numpy())
check("F test recall at 1,082 alerts", f"{pc['recall']:.4f}", "0.8846")
check("F test dollar recall at 1,082 alerts", f"{pc['dollar_recall']:.4f}", "0.9494")
THRESHOLDS = [0.5, 0.2940968658, 0.1, 0.02, 0.005]
TH = threshold_table(tx_exp, THRESHOLDS, DAYS).set_index("threshold")
for thr, (per_day, fraud_tx, cards) in {0.1: ("14.6", 860, 360), 0.02: ("27.8", 899, 590)}.items():
    r = TH.loc[thr]
    check(f"threshold {thr}: step-ups per day", f"{r['stepups_per_day']:.1f}", per_day)
    check(f"threshold {thr}: fraud transactions stepped up", int(r["fraud_tx"]), fraud_tx)
    check(f"threshold {thr}: cards touched", int(r["cards_touched"]), cards)

cards_all = np.array(sorted(tx["card_key"].unique()))
exp_c = per_card_metrics(tx_exp, 0.1, cards_all)
pre_c = per_card_metrics(tx_pre, 0.1, cards_all)
active = (exp_c["n_tx"] > 0).to_numpy()
Y = exp_c.loc[active, "fraud_loss"]
check("active cards in the experiment period", int(active.sum()), 915)
check("per-card fraud loss: mean", f"{Y.mean():.2f}", "535.40")
check("per-card fraud loss: SD", f"{Y.std():.2f}", "1722.67")
check("per-card fraud loss: share non-zero", f"{(Y > 0).mean():.3f}", "0.103")
check("cards per arm for a 10% MDE (all cards)", f"{required_n(Y.std(), 0.1 * Y.mean()):,.0f}", "16,251")
check("CUPED, primary (pre max F score)",
      f"{variance_reduction(Y, pre_c.loc[active, 'max_score']):.1%}", "1.9%")
check("CUPED, guardrail base (pre legit $ above 0.1)",
      f"{variance_reduction(exp_c.loc[active, 'legit_above'], pre_c.loc[active, 'legit_above']):.1%}", "15.1%")
if problems:
    raise RuntimeError(f"reconciliation failed, stopping: {problems}")
print("\ndeterministic reconciliation passed")

In [ ]:
# Reconciliation 2 (simulated rates: must agree within Monte Carlo error; seeds differ from the design script)
from src.experiments.ab_design import peeking_fpr, welch_analysis

def mc_ok(label, got, ref, n):
    tol = 3 * np.sqrt(ref * (1 - ref) / n)
    ok = abs(got - ref) <= tol
    print(f"{'ok  ' if ok else 'FAIL'} {label}: {got:.3f} here vs {ref:.3f} in the design "
          f"(difference {got - ref:+.3f}, allowed ±{tol:.3f} = 3 Monte Carlo SE)")
    if not ok:
        problems.append(label)

y_all = Y.to_numpy()
rng = np.random.default_rng(MASTER_SEED)
aa = [welch_analysis(y_all, rng.permutation(len(y_all)) < len(y_all) // 2)["p"] < 0.05 for _ in range(1000)]
mc_ok("A/A false-positive rate, all active cards, Welch", float(np.mean(aa)), 0.045, 1000)

week = ((tx_exp["ts"] - pd.Timestamp("2020-10-01")).dt.days // 7).clip(upper=12)
weekly = (tx_exp.assign(week=week, f=tx_exp["amt"] * tx_exp["is_fraud"])
          .pivot_table(index="card_key", columns="week", values="f", aggfunc="sum", fill_value=0))
cum = weekly.reindex(cards_all[active]).fillna(0).cumsum(axis=1).to_numpy()
PEEK = peeking_fpr(cum, n_sims=1000, seed=MASTER_SEED + 1)
mc_ok("peeking: stop at the first p < 0.05 over 13 weekly looks", PEEK["any_look"], 0.211, 1000)
mc_ok("peeking: fixed 92-day horizon", PEEK["fixed_horizon"], 0.056, 1000)
if problems:
    raise RuntimeError(f"reconciliation failed, stopping: {problems}")
print("\nsimulated reconciliation passed (the design used different random draws, so exact equality is not expected)")

In [ ]:
# b) Step-up thresholds in the experiment period, and the triggered population at each
from src.experiments.simulation import prepare_stepup

STEPUP = {thr: prepare_stepup(tx_exp, thr, cards_all[active]) for thr in THRESHOLDS}
TH["triggered_cards"] = [int(STEPUP[t].triggered.sum()) for t in TH.index]
TH["fraud_loss_on_triggered_cards"] = [STEPUP[t].fraud_loss[STEPUP[t].triggered].sum() / STEPUP[t].fraud_loss.sum()
                                       for t in TH.index]
show = TH[["stepups_per_day", "fraud_tx", "fraud_dollar_share", "legit_tx_per_day", "legit_dollars",
           "legit_dollar_share", "cards_touched", "triggered_cards", "fraud_loss_on_triggered_cards"]]
print(show.round(4).to_string())
PRIMARY_THR = 0.02
DATA = STEPUP[PRIMARY_THR]
print(f"\nprimary threshold {PRIMARY_THR}: {DATA.triggered.sum()} triggered of {len(DATA.cards)} active cards")

In [ ]:
# d) Power and sample size from the real per-card distribution: all enrolled vs triggered, as-is vs rescaled
from scipy import stats

from src.experiments.ab_design import mde
from src.experiments.simulation import resample_population

pop = {"all enrolled": DATA.fraud_loss, "triggered (0.02)": DATA.fraud_loss[DATA.triggered]}
rows = []
for name, v in pop.items():
    row = {"population": name, "cards": len(v), "mean": v.mean(), "sd": v.std(ddof=1),
           "share_nonzero": (v > 0).mean(), "MDE_with_these_cards": mde(v.std(ddof=1), len(v) // 2) / v.mean()}
    for r in (0.05, 0.10, 0.20, 0.50):
        row[f"n_per_arm_{r:.0%}"] = required_n(v.std(ddof=1), r * v.mean())
    rows.append(row)
nonzero = DATA.fraud_loss[DATA.fraud_loss > 0]
for inc in (0.01, 0.005):
    mu = inc * nonzero.mean()
    sd = np.sqrt(inc * (nonzero ** 2).mean() - mu ** 2)
    row = {"population": f"all enrolled, rescaled to {inc:.1%} incidence", "cards": np.nan, "mean": mu, "sd": sd,
           "share_nonzero": inc, "MDE_with_these_cards": np.nan}
    for r in (0.05, 0.10, 0.20, 0.50):
        row[f"n_per_arm_{r:.0%}"] = required_n(sd, r * mu)
    rows.append(row)
SIZING = pd.DataFrame(rows).set_index("population")
print(SIZING.round(3).to_string())

# Resampling check: at the formula's n for a 10% effect, simulated power should be close to 0.8
rng = np.random.default_rng(MASTER_SEED + 2)
for label, inc in [("all enrolled, as-is", None), ("all enrolled, 1% incidence", 0.01)]:
    key = "all enrolled" if inc is None else f"all enrolled, rescaled to {inc:.1%} incidence"
    n = int(np.ceil(SIZING.loc[key, "n_per_arm_10%"]))
    hits = [stats.ttest_ind(resample_population(DATA.fraud_loss, n, rng, inc),
                            0.9 * resample_population(DATA.fraud_loss, n, rng, inc), equal_var=False).pvalue < 0.05
            for _ in range(200)]
    print(f"resampling check, {label}: {n:,} cards per arm -> simulated power {np.mean(hits):.2f} (formula: 0.80)")

In [ ]:
# e) CUPED: correlation between a pre-period covariate and the experiment-period metric (variance reduction = rho^2)
pre_02 = per_card_metrics(tx_pre, PRIMARY_THR, cards_all[active])
exp_02 = per_card_metrics(tx_exp, PRIMARY_THR, cards_all[active])
trig = DATA.triggered
rows = []
for pop_name, m in [("all enrolled", np.ones(len(trig), dtype=bool)), ("triggered (0.02)", trig)]:
    for metric, y_col, x_col in [("fraud loss (primary)", "fraud_loss", "max_score"),
                                 ("fraud loss (primary)", "fraud_loss", "legit_spend"),
                                 ("legit $ stepped up (guardrail base)", "legit_above", "legit_above"),
                                 ("legit spend (secondary)", "legit_spend", "legit_spend")]:
        y, x = exp_02[y_col].to_numpy()[m], pre_02[x_col].to_numpy()[m]
        rows.append({"population": pop_name, "metric": metric, "pre-period covariate": x_col,
                     "rho": np.corrcoef(y, x)[0, 1], "variance_reduction": variance_reduction(y, x)})
CUPED_TABLE = pd.DataFrame(rows)
print(CUPED_TABLE.round(3).to_string(index=False))
print("\nThe primary metric gains almost nothing: each card has one fraud episode, so past risk does not predict"
      " future fraud loss here. CUPED is used for the guardrail and secondary metrics only.")

In [ ]:
# g) Full-pipeline A/A (the real assignment, step-up and analysis code with no effect): stop conditions
from src.experiments.ab_design import srm_pvalue
from src.experiments.simulation import assign_arms, simulate_experiments

AA = simulate_experiments(DATA, 0.0, 0.0, 1.0, n_sims=1000, seed=MASTER_SEED + 3)
aa_fpr = (AA["primary_p"] < 0.05).mean()
aa_ship = (AA["decision"] == "ship").mean()
designated = assign_arms(len(DATA.cards), np.random.default_rng(MASTER_SEED))
srm_designated = srm_pvalue(int(designated.sum()), int((~designated).sum()))
AA_ITT = simulate_experiments(DATA, 0.0, 0.0, 1.0, n_sims=1000, seed=MASTER_SEED + 4, population="all")
print(f"A/A, triggered cards, 1,000 simulated experiments: false-positive rate {aa_fpr:.3f} (required 0.035-0.065)")
print(f"A/A, all enrolled (ITT): false-positive rate {(AA_ITT['primary_p'] < 0.05).mean():.3f}")
print(f"A/A: share with SRM p < 0.001 {(AA['srm_p'] < 0.001).mean():.3f}; share 'ship' {aa_ship:.3f}")
print(f"designated A/A assignment (seed {MASTER_SEED}): {designated.sum()} treatment vs {(~designated).sum()} control, "
      f"SRM p = {srm_designated:.3f}")
if not 0.035 <= aa_fpr <= 0.065:
    raise RuntimeError(f"A/A false-positive rate {aa_fpr:.3f} outside 0.035-0.065: stopping")
if srm_designated < 0.001:
    raise RuntimeError("SRM in the designated A/A: stopping")
print("A/A checks passed")

In [ ]:
# g) Peeking and O'Brien-Fleming boundaries (illustration only: the experiment uses a fixed 92-day horizon)
from src.experiments.ab_design import obf_boundaries

print(f"A/A with 13 weekly looks: false-positive rate {PEEK['any_look']:.3f} when stopping at the first p < 0.05, "
      f"{PEEK['fixed_horizon']:.3f} at the fixed horizon")
for k in (3, 13):
    b = obf_boundaries(k, alpha=0.05, n_sims=200_000, seed=MASTER_SEED + 5)
    print(f"O'Brien-Fleming z boundaries for {k} looks: " + ", ".join(f"{v:.2f}" for v in b))
print("A sequential design would stop early only if |z| crossed the boundary at that look; with a fixed horizon"
      " there is one look and the usual 1.96.")

In [ ]:
# h) Break-even abandonment rate: blocked fraud dollars = lost legitimate revenue (b and m are assumptions)
from src.experiments.ab_design import breakeven_abandonment

rows = []
for thr in THRESHOLDS:
    r = TH.loc[thr]
    row = {"threshold": round(thr, 4)}
    for b in (0.5, 0.65, 0.8):
        for m in (1.0, 0.25, 0.02):
            row[f"b={b:.0%} m={m:.0%}"] = breakeven_abandonment(r["fraud_dollars"], r["legit_dollars"], b, m)
    rows.append(row)
BREAKEVEN = pd.DataFrame(rows).set_index("threshold")
print("break-even abandonment rate (above 100% means step-up pays even if every stepped-up customer abandons):")
print(BREAKEVEN.map(lambda v: f"{v:.0%}").to_string())

## Simulated experiments (SIMULATED effects)

Each simulated experiment re-runs the whole pipeline: coin-flip assignment of the 915 active cards, the simulated step-up on treated cards (fraud at or above the threshold blocked with probability b, legitimate transactions abandoned with probability a), and the pre-registered analysis and decision. The true effects are known exactly from the assumptions, so the estimates can be checked against them.

In [ ]:
# i, j) Primary scenarios: threshold 0.02, triggered cards, m = 1.0, 500 simulated experiments each
from src.experiments.simulation import true_effects

def scenario_rows(data, grid, m, n_sims, seed, population="triggered"):
    pop_mask = data.triggered if population == "triggered" else np.ones(len(data.cards), dtype=bool)
    rows = []
    for i, (b, a) in enumerate(grid):
        sims = simulate_experiments(data, b, a, m, n_sims=n_sims, seed=seed + i, population=population)
        truth = true_effects(data, b, a, m, pop_mask)
        rows.append({"b": b, "a": a, "true_fraud_reduction": truth["fraud_loss_reduction"],
                     "mean_estimate": sims["primary_diff"].mean(), "primary_power": (sims["primary_p"] < 0.05).mean(),
                     "true_total_reduction": truth["total_loss_reduction"],
                     "total_ci_above_0": (sims["total_ci_low"] > 0).mean(),
                     "g1_breach": (sims["g1_p_holm"] < 0.05).mean(), "g2_breach": (sims["g2_p_holm"] < 0.05).mean(),
                     "ship": (sims["decision"] == "ship").mean(), "decision": sims["decision"].mode()[0]})
    return pd.DataFrame(rows)

GRID = [(b, a) for b in (0.5, 0.65, 0.8) for a in (0.01, 0.05, 0.10, 0.20)]
t0 = time.perf_counter()
PRIMARY = scenario_rows(DATA, GRID, 1.0, n_sims=500, seed=MASTER_SEED + 100)
print(f"{len(GRID)} scenarios x 500 simulated experiments in {time.perf_counter() - t0:.0f} s "
      f"({DATA.triggered.sum()} triggered cards per experiment; dollars per triggered card)")
print(PRIMARY.round(3).to_string(index=False))
bias = (PRIMARY["mean_estimate"] - PRIMARY["true_fraud_reduction"]) / PRIMARY["true_fraud_reduction"]
print(f"\nestimate vs truth: largest relative bias {bias.abs().max():.1%} (unbiased up to Monte Carlo error)")

In [ ]:
# Secondary checks: all enrolled (ITT), lost-revenue share m, and other thresholds (sensitivity only)
t0 = time.perf_counter()
ITT = scenario_rows(DATA, GRID, 1.0, n_sims=300, seed=MASTER_SEED + 200, population="all")
print("all enrolled cards (ITT), m = 1.0, 300 simulated experiments each:")
print(ITT[["b", "a", "true_fraud_reduction", "primary_power", "total_ci_above_0", "ship", "decision"]].round(3)
      .to_string(index=False))
print("\nprimary power, triggered vs all enrolled:",
      ", ".join(f"b={b:.0%}: {p:.2f} vs {q:.2f}" for b, p, q in
                zip(PRIMARY["b"][::4], PRIMARY["primary_power"][::4], ITT["primary_power"][::4])), "(at a = 1%)")

M_SENS = pd.concat([scenario_rows(DATA, [(0.65, a) for a in (0.05, 0.10, 0.20)], m, n_sims=300,
                                  seed=MASTER_SEED + 300 + int(m * 100)).assign(m=m) for m in (0.25, 0.02)])
print("\nlost-revenue share m = 0.25 and 0.02 (b = 65%):")
print(M_SENS[["m", "a", "true_total_reduction", "total_ci_above_0", "g1_breach", "g2_breach", "ship", "decision"]]
      .round(3).to_string(index=False))

THR_SENS = pd.concat([scenario_rows(STEPUP[t], [(0.65, 0.05)], 1.0, n_sims=300, seed=MASTER_SEED + 400 + i)
                      .assign(threshold=round(t, 4), triggered=int(STEPUP[t].triggered.sum()))
                      for i, t in enumerate(THRESHOLDS)])
print("\nother thresholds (b = 65%, a = 5%, m = 1.0):")
print(THR_SENS[["threshold", "triggered", "true_fraud_reduction", "primary_power", "true_total_reduction",
                "total_ci_above_0", "ship", "decision"]].round(3).to_string(index=False))
print(f"secondary checks in {time.perf_counter() - t0:.0f} s")

In [ ]:
# Novelty effect: a design element whose result is NULL BY CONSTRUCTION here
rng = np.random.default_rng(MASTER_SEED + 6)
treat_cards = set(DATA.cards[assign_arms(len(DATA.cards), rng)])
exposed = tx_exp[(tx_exp["score"] >= PRIMARY_THR) & (tx_exp["is_fraud"] == 0) & tx_exp["card_key"].isin(treat_cards)]
first = exposed.groupby("card_key")["ts"].transform("min")
weeks_since_first = ((exposed["ts"] - first).dt.days // 7).clip(upper=8)
abandoned = rng.random(len(exposed)) < 0.05
novelty = pd.DataFrame({"week": weeks_since_first, "abandoned": abandoned}).groupby("week")["abandoned"].agg(["size", "mean"])
print("abandonment by week since a card's first step-up, one simulated experiment (a = 5%):")
print(novelty.rename(columns={"size": "stepped_up_legit_tx", "mean": "abandonment_rate"}).round(3).to_string())
print("\nThe simulator applies the same abandonment rate every week, so there is no novelty effect to find:"
      " this check is part of the design (in a real experiment, compare early weeks with later ones, and"
      " exclude each card's first 14 days as a sensitivity analysis), not a finding.")

## Part 2: observational methods against a known truth (semi-synthetic, SIMULATED)

The real outcomes cannot show confounding usefully: pre-period risk is *negatively* related to fraud loss in the experiment period (one fraud episode per card), and only 94 cards have fraud. So the main analysis is **semi-synthetic**, in the style of the IHDP benchmark: real per-card pre-period covariates, a simulated fraud outcome (loss sizes drawn from the real non-zero losses), a simulated rollout that targets riskier cards, and a heterogeneous true effect. Both potential outcomes are known for every card, so every estimand is exact.

- **Each estimator is compared with its own estimand.** Matching with a caliper estimates the effect on the matched treated cards; inverse probability weighting with trimming (0.05–0.95) estimates the effect on the trimmed population; the doubly robust and regression estimators target the average effect (ATE).
- **Matching inference.** Bias and RMSE only. A naive bootstrap is invalid for matching estimators (Abadie & Imbens 2008), and the Abadie–Imbens standard error for an estimated propensity score is out of scope. Caliper: 0.2 standard deviations of the logit of the propensity score.
- **Gender** is never used by the rollout or the outcome; it appears only in the balance check.
- **Rejected alternative:** a "natural" causal question inside the simulated data has no ground truth, so the methods could not be checked.

In [ ]:
# Real per-card covariates (pre-period) and the small REAL-outcome version: confounding is weak and inverted
from src.experiments.observational import fit_propensity, ipw, naive_diff

both = active & (pre_c["n_tx"].to_numpy() > 0)
cards_both = cards_all[both]
pre_b = per_card_metrics(tx_pre, PRIMARY_THR, cards_both)
card_info = tx_exp.groupby("card_key")[["age_band", "urban", "gender"]].first().reindex(cards_both)
COV = pd.DataFrame({"night_share": pre_b["night_share"], "max_score": pre_b["max_score"],
                    "log_spend": np.log1p(pre_b["legit_spend"]), "age_band": card_info["age_band"],
                    "urban": card_info["urban"]}).reset_index(drop=True)
GENDER_F = (card_info["gender"] == "F").astype(float).to_numpy()
exp_b = per_card_metrics(tx_exp, PRIMARY_THR, cards_both)
y0 = exp_b["fraud_loss"].to_numpy()
LOSS_SIZES = y0[y0 > 0]
print(f"{len(cards_both)} cards active in both periods; {len(LOSS_SIZES)} with fraud in the experiment period")
print(f"correlation of pre-period max F score with experiment fraud loss: {np.corrcoef(COV['max_score'], y0)[0, 1]:+.3f}")

b_real = 0.65                                         # SIMULATED share of stepped-up fraud blocked
effect_real = b_real * exp_b["fraud_above"].to_numpy()  # true reduction per card if treated
z = (COV["max_score"] - COV["max_score"].mean()) / COV["max_score"].std()
rng = np.random.default_rng(MASTER_SEED + 7)
rows = []
for _ in range(500):
    t = rng.random(len(z)) < 1 / (1 + np.exp(-(-0.85 + 1.5 * z.to_numpy())))  # policy: riskier cards more likely
    y = np.where(t, y0 - effect_real, y0)
    ps = fit_propensity(COV[["max_score"]], t)
    rows.append({"treated_y0": y0[t].mean(), "control_y0": y0[~t].mean(), "naive": -naive_diff(y, t),
                 "ipw_ate": -ipw(y, t, ps, "ate", trim=None)["est"], "true_att": effect_real[t].mean()})
REAL = pd.DataFrame(rows)
print(f"real outcomes, 500 simulated rollouts: treated cards' own loss without treatment ${REAL.treated_y0.mean():.0f} "
      f"vs control ${REAL.control_y0.mean():.0f} (the policy picks LOWER-loss cards: inverted confounding)")
print(f"true ATE ${effect_real.mean():.0f} | true ATT ${REAL.true_att.mean():.0f} | naive ${REAL.naive.mean():.0f} | "
      f"IPW ${REAL.ipw_ate.mean():.0f} (SD across rollouts ${REAL.ipw_ate.std():.0f})")

In [ ]:
# Overlap and covariate balance for one semi-synthetic rollout (20,000 cards drawn from the real covariates)
from src.experiments.observational import make_semi_synthetic, overlap_summary, psm_att, smd

d = make_semi_synthetic(COV, 20_000, LOSS_SIZES, seed=MASTER_SEED + 8)
X = d["X"][list(COV.columns)]
ps = fit_propensity(X, d["t"])
print(f"semi-synthetic rollout: {d['t'].mean():.1%} of cards treated, {(d['y0'] > 0).mean():.1%} with fraud; "
      f"true ATE ${d['ate']:.0f}, true ATT ${d['att']:.0f} (they differ: the effect is heterogeneous)")
print("\npropensity-score overlap:")
print(overlap_summary(ps, d["t"]).round(3).to_string())

w = np.where(d["t"], 1 / ps, 1 / (1 - ps))
m = psm_att(d["y"], d["t"], ps)
mt = np.flatnonzero(m["matched_treated"])
Xm = pd.concat([X.iloc[mt], X.iloc[m["controls"]]], ignore_index=True)
tm = np.r_[np.ones(len(mt), dtype=bool), np.zeros(len(m["controls"]), dtype=bool)]
gender_sim = np.random.default_rng(MASTER_SEED + 9).choice(GENDER_F, len(X))  # not used by rollout or outcome
balance = pd.DataFrame({"before": smd(X.assign(gender_F=gender_sim), d["t"]),
                        "after IPW": smd(X.assign(gender_F=gender_sim), d["t"], w),
                        "after matching": smd(Xm.assign(gender_F=0.0), tm)})
balance.loc["gender_F", "after matching"] = np.nan
print("\nstandardised mean differences (target |SMD| < 0.1; gender is a balance check only):")
print(balance.round(3).to_string())
print(f"\nmatching: caliper {m['caliper']:.3f} on the logit scale; {m['n_unmatched']} of {d['t'].sum()} treated unmatched")

In [ ]:
# Bias, RMSE and coverage: each estimator against its own estimand, 200 rollouts x 20,000 cards per scenario
from src.experiments.observational import compare_estimators, summarise_estimators

t0 = time.perf_counter()
SCEN = {
    "measured confounders": dict(),
    "hidden confounder (night share not observed)": dict(observed=["max_score", "log_spend", "age_band", "urban"]),
    "randomised experiment (benchmark)": dict(randomised=True),
}
PART2 = {}
for name, kw in SCEN.items():
    res = compare_estimators(COV, LOSS_SIZES, n=20_000, n_reps=200, seed=MASTER_SEED + 1000, **kw)
    PART2[name] = summarise_estimators(res)
    print(f"\n=== {name} ===")
    print(PART2[name].round(3).to_string())
print(f"\nPart 2 in {time.perf_counter() - t0:.0f} s. Coverage is blank where no valid standard error is computed"
      " (matching: see above; regression adjustment: bias and RMSE only).")

In [ ]:
# Final checks: mlflow.db unchanged, runtime within the limit
mlflow_sha_after = sha256(MLFLOW_DB)
runtime = time.perf_counter() - T_START
print("mlflow.db SHA-256 before:", MLFLOW_SHA_BEFORE)
print("mlflow.db SHA-256 after: ", mlflow_sha_after)
assert mlflow_sha_after == MLFLOW_SHA_BEFORE, "mlflow.db changed: stopping"
print(f"runtime {runtime / 60:.1f} minutes")
assert runtime < 30 * 60, "runtime above 30 minutes: stopping"
print("all checks passed; every treatment effect above is a SIMULATED ASSUMPTION")